# **Logistic Regression: Stroke Predictions** #

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\Lailah\\vscode-project\\DA2_Assessment\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\Lailah\\vscode-project\\DA2_Assessment'

# Section 1

Section 1 content

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [29]:
df = pd.read_csv("Data/Cleaned_Data/Cleaned_Data.csv")
df.head(8)

,hypertension,heart_disease,bmi,weight_category,avg_glucose_level,stroke,gender,gender_Male,gender_Other,age,ever_married,ever_married_Yes,Residence_type,Residence_type_Urban,work_type,work_type_encoded
0,0,1,36.6,Obese,169.3575,1,Male,1.0,0.0,67.0,Yes,1.0,Urban,1.0,Private,2.0
1,0,0,28.1,Overweight,169.3575,1,Female,0.0,0.0,61.0,Yes,1.0,Rural,0.0,Self-employed,3.0
2,0,1,32.5,Obese,105.9200,1,Male,1.0,0.0,80.0,Yes,1.0,Rural,0.0,Private,2.0
3,0,0,34.4,Obese,169.3575,1,Female,0.0,0.0,49.0,Yes,1.0,Urban,1.0,Private,2.0
4,1,0,24.0,Healthy,169.3575,1,Female,0.0,0.0,79.0,Yes,1.0,Rural,0.0,Self-employed,3.0
5,0,0,29.0,Overweight,169.3575,1,Male,1.0,0.0,81.0,Yes,1.0,Urban,1.0,Private,2.0
6,1,1,27.4,Overweight,70.0900,1,Male,1.0,0.0,74.0,Yes,1.0,Rural,0.0,Private,2.0
7,0,0,22.8,Healthy,94.3900,1,Female,0.0,0.0,69.0,No,0.0,Urban,1.0,Private,2.0


In [30]:
df.columns

Index(['hypertension', 'heart_disease', 'bmi', 'weight_category',
       'avg_glucose_level', 'stroke', 'gender', 'gender_Male', 'gender_Other',
       'age', 'ever_married', 'ever_married_Yes', 'Residence_type',
       'Residence_type_Urban', 'work_type', 'work_type_encoded'],
      dtype='object')

Removed all columns that have string data as this is not compatible with Logistic Regression model. This isnt an issues as all these variables have been encoded so their value is still present

In [31]:
df = df.drop(
    columns=[
        "gender",
        "ever_married",
        "work_type",
        "Residence_type",
        'weight_category',
    ],
    #errors="ignore",
)
df.head()

,hypertension,heart_disease,bmi,avg_glucose_level,stroke,gender_Male,gender_Other,age,ever_married_Yes,Residence_type_Urban,work_type_encoded
0,0,1,36.6,169.3575,1,1.0,0.0,67.0,1.0,1.0,2.0
1,0,0,28.1,169.3575,1,0.0,0.0,61.0,1.0,0.0,3.0
2,0,1,32.5,105.9200,1,1.0,0.0,80.0,1.0,0.0,2.0
3,0,0,34.4,169.3575,1,0.0,0.0,49.0,1.0,1.0,2.0
4,1,0,24.0,169.3575,1,0.0,0.0,79.0,1.0,0.0,3.0


Split dataset into train and test sets. Random state was selcted as 101 for preproducability purposes

In [60]:
# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     df.drop(["stroke"],axis=1), df["stroke"], test_size=0.2, random_state=101,stratify=df["stroke"]
# )

# print(
#     "* Train set:",
#     X_train.shape,
#     y_train.shape,
#     "\n* Test set:",
#     X_test.shape,
#     y_test.shape,
# )

In [61]:
from sklearn.model_selection import train_test_split

X = df.drop("stroke", axis=1)
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [62]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


**Create a pipeline - Binary Classifier (Logistic Regression)**

In [ ]:
# from sklearn.pipeline import Pipeline

# ##Feature Scaling 
# from sklearn.preprocessing import StandardScaler

# ##Feature Selection
# from sklearn.feature_selection import SelectFromModel

# ##ML Algorithms
# from sklearn.linear_model import LogisticRegression

# def pipeline_logistic_regression():
#     pipeline = Pipeline(
#         [
#             ("scaler", StandardScaler()),
#             ("feature_selection", SelectFromModel(LogisticRegression(random_state=101))),
#             ("model", LogisticRegression()),
#         ]
#     )
#     return pipeline

In [63]:
def pipeline_logistic_regression():
    pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            
            # Feature selection using a balanced logistic regression
            ("feature_selection", 
             SelectFromModel(
                 LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
             )
            ),

            # Final model also using balanced logistic regression
            ("model", 
             LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
            ),
        ]
    )
    return pipeline


**Fitting the pipeline to the train set**

In [64]:
pipeline=pipeline_logistic_regression()
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('feature_selection',
                 SelectFromModel(estimator=LogisticRegression(class_weight='balanced',
                                                              max_iter=1000,
                                                              random_state=42))),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

**Evaluate Pipeline**

In [65]:
def logistic_regression_coef(model,columns):
    coeff_df=pd.DataFrame(
        model.coef_, index=["Coefficient"], columns=columns
    ).T.sort_values(["Coefficient"], key=abs, ascending=False)
    print(coeff_df)

In [66]:
logistic_regression_coef(
    model=pipeline["model"],
    columns=X_train.columns[pipeline["feature_selection"].get_support()],
)

     Coefficient
age     1.864323


In [67]:
from sklearn.metrics import classification_report, confusion_matrix
def confusion_matrix_and_report(X,y, pipeline, label_map):
    prediction=pipeline.predict(X)

    print("--- Confusion Matrix ---")
    print(
        pd.DataFrame(
            confusion_matrix(y_true=prediction, y_pred=y),
            columns=[["Actual" + sub for sub in label_map]], 
            index=[["Prediction" + sub for sub in label_map]],
        )
    )
    print("\n")

    print("--- Classification Report ---")
    print(classification_report(y, prediction, target_names=label_map), "\n")

In [68]:
def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    print("###Train set performance###\n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("###Test set performance###\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [69]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline,
    label_map=["No Stroke", "Stroke"],
)

###Train set performance###

--- Confusion Matrix ---
                    ActualNo Stroke ActualStroke
PredictionNo Stroke            2818           44
PredictionStroke               1071          155


--- Classification Report ---
              precision    recall  f1-score   support

   No Stroke       0.98      0.72      0.83      3889
      Stroke       0.13      0.78      0.22       199

    accuracy                           0.73      4088
   macro avg       0.56      0.75      0.53      4088
weighted avg       0.94      0.73      0.80      4088
 

###Test set performance###

--- Confusion Matrix ---
                    ActualNo Stroke ActualStroke
PredictionNo Stroke             715            9
PredictionStroke                257           41


--- Classification Report ---
              precision    recall  f1-score   support

   No Stroke       0.99      0.74      0.84       972
      Stroke       0.14      0.82      0.24        50

    accuracy                           0.7

---

# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
